In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

In [2]:
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

In [5]:
examples = [
    "I love this product, it's amazing!",
    "This is terrible, I'm very disappointed.",
    "It's okay, nothing special."
]

In [6]:
for text in examples:
    result = classifier(text)
    print(f"Text: {text}")
    print(f"Label: {result[0]['label']} | Score: {result[0]['score']:.2f}\n")

Text: I love this product, it's amazing!
Label: positive | Score: 0.99

Text: This is terrible, I'm very disappointed.
Label: negative | Score: 0.94

Text: It's okay, nothing special.
Label: neutral | Score: 0.60



In [7]:
from datasets import load_dataset
dataset = load_dataset("tweet_eval", "sentiment")

In [8]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [10]:
dataset['train'][0]

{'text': '"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"',
 'label': 2}

In [17]:
from sklearn.metrics import classification_report

In [18]:
dataset_label_map  = {0: "negative", 1: "neutral", 2: "positive"}

In [19]:
# Prendi un campione (es. 500 esempi per velocità)
test_sample = dataset["test"]

In [20]:
y_true = [dataset_label_map[l] for l in test_sample["label"]]
y_pred = [classifier(t)[0]["label"].lower() for t in test_sample["text"]]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [21]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

    negative       0.69      0.81      0.74      3972
     neutral       0.76      0.66      0.70      5937
    positive       0.71      0.74      0.73      2375

    accuracy                           0.72     12284
   macro avg       0.72      0.73      0.72     12284
weighted avg       0.73      0.72      0.72     12284



In [22]:
print(model.config.id2label)

{0: 'negative', 1: 'neutral', 2: 'positive'}


In [13]:
model.save_pretrained("../models/sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
tokenizer.save_pretrained("../models/sentiment_model")

('../models/sentiment_model\\tokenizer_config.json',
 '../models/sentiment_model\\tokenizer.json')